# Part 2: Transfer LearningUse ResNet50 pretrained on ImageNet for landmark classification.**Target accuracy: ≥60%**

In [ ]:
import torchimport torch.nn as nnimport torch.optim as optimfrom torch.optim.lr_scheduler import ReduceLROnPlateauimport syssys.path.insert(0, '.')from src.data import get_data_loadersfrom src.transfer import get_pretrained_modelfrom src.optimization import get_lossfrom src.train import train_model, test_model, plot_training_historyfrom src.helpers import get_device, count_parameters

## 1. Load Data

In [ ]:
DATA_DIR = "landmark_images"BATCH_SIZE = 64data_loaders, image_datasets = get_data_loaders(DATA_DIR, batch_size=BATCH_SIZE)class_names = image_datasets["train"].classesprint(f"Number of classes: {len(class_names)}")

## 2. Load Pretrained ResNet50Freeze all layers except the new classification head: `Linear(2048, 50)`.**Why ResNet50:**- Residual connections solve gradient vanishing- ImageNet features provide strong visual representations- 2048-dim feature vector is rich enough for 50 classes

In [ ]:
device = get_device()print(f"Using device: {device}")model = get_pretrained_model(num_classes=50).to(device)count_parameters(model)

## 3. Training SetupOnly train the new fc layer (much faster than from scratch).

In [ ]:
criterion = get_loss()optimizer = optim.Adam(model.fc.parameters(), lr=0.001)scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, verbose=True)

## 4. Train (15 epochs)

In [ ]:
history = train_model(    model, data_loaders, optimizer, criterion, scheduler,    device=device, n_epochs=15, save_path="transfer_best.pth")

## 5. Training Curves

In [ ]:
plot_training_history(history)

## 6. Test the Model

In [ ]:
model.load_state_dict(torch.load("transfer_best.pth", map_location=device))model.to(device)test_acc = test_model(model, data_loaders["test"], device)print(f"\nTarget: ≥60% | Achieved: {test_acc:.2f}%")assert test_acc >= 60, f"Test accuracy {test_acc:.2f}% is below the 60% target!"